In [ ]:
# ==============================================================================
# GLINKER — Medical Intake Pipeline
# main.ipynb  (orchestration only — all logic lives in glinker/ and pipeline.py)
#
# Run cells top-to-bottom on first use.
# On subsequent runs, skip Cell 5 (corpus already indexed) unless you want
# to rebuild the knowledge base from scratch.
# ==============================================================================


In [ ]:
# ── Cell 1: Pull code from GitHub ────────────────────────────────────────────
#
# The repo is cloned into /kaggle/working/curesense-project/ on first run.
# On later runs in the same session it does a git pull to get latest changes.
# REPO_DIR is added to sys.path so glinker/, pipeline.py, and api/ are
# importable immediately after the clone.
#
import sys, os, subprocess

REPO_URL  = 'https://github.com/moizaimran/curesense-project.git'
BRANCH    = 'hassan-branch'
REPO_DIR  = '/kaggle/working/curesense-project'

# If your repo is PRIVATE: add a Kaggle secret named 'GH_TOKEN'
# (Settings -> Secrets -> Add new secret, paste a GitHub Personal Access Token)
# and uncomment the two lines below:
# from kaggle_secrets import UserSecretsClient
# _token   = UserSecretsClient().get_secret('GH_TOKEN')
# REPO_URL = REPO_URL.replace('https://', f'https://{_token}@')

if not os.path.exists(REPO_DIR):
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)
print('Ready:', REPO_DIR)


In [ ]:
# ── Cell 2: Installs ──────────────────────────────────────────────────────────
# requirements.txt covers Flask service deps (pdfplumber, pypdfium2, openai,
# whisper, gliner, faiss, etc.)
!pip install -r {REPO_DIR}/requirements.txt -q

# Pin transformers >= 4.40 — required for AutoModelForImageTextToText (MedGemma).
# openai-whisper in requirements.txt can pull an older version; this overrides it.
!pip install -q "transformers>=4.40.0"

# accelerate — required for device_map= (will hard-crash without it)
# pyngrok    — used in Cell 3 and Cell 9 (not in requirements.txt)
# fastapi / uvicorn / pydicom — MedGemma FastAPI service
# httpx      — async HTTP client used by the reverse proxy in Cell 9
!pip install -q accelerate pyngrok fastapi uvicorn pydicom httpx

!pip install torchvision --upgrade --quiet

In [ ]:
# ── Cell 3: Secrets + OpenAI client + HuggingFace login ──────────────────────
#
# Kaggle secrets required (Settings → Secrets → Add new secret):
#   'Ngrok Key'   — ngrok auth token
#   'OpenAI Key'  — OpenAI API key
#   'HF_TOKEN'    — HuggingFace token with access to google/medgemma-1.5-4b-it
#                   (accept the model terms at hf.co/google/medgemma-1.5-4b-it first)
#
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
from openai import OpenAI
from huggingface_hub import login
import glinker.config as cfg

_secrets = UserSecretsClient()

ngrok.set_auth_token(_secrets.get_secret('Ngrok Key'))
cfg.openai_client = OpenAI(api_key=_secrets.get_secret('OpenAI Key'))

# HuggingFace login — required to download the gated MedGemma model
login(token=_secrets.get_secret('HF_TOKEN'), add_to_git_credential=False)

print('OpenAI client ready')
print('HuggingFace login OK')

In [ ]:
# ── Cell 4: Load heavy models (Whisper + GLiNER) ──────────────────────────────
import torch
import whisper
from gliner import GLiNER
import glinker.config as cfg

print('Loading Whisper Large ...')
cfg.whisper_model = whisper.load_model('large', device='cuda')
print('Whisper ready')

print('Loading GLiNER BioMed ...')
cfg.gliner_model = GLiNER.from_pretrained('Ihor/gliner-biomed-bi-large-v1.0').to('cuda')
print('GLiNER ready on:', next(cfg.gliner_model.parameters()).device)


In [ ]:
# ── Cell 5: Load or build the RAG index ──────────────────────────────────────
#
# Priority order:
#   1. /kaggle/working/rag_index/                                    — already built this session
#   2. /kaggle/input/datasets/hassanraheem/uresense-rag-index/       — pre-built dataset
#   3. Build from scratch                                             — first time only
#
import os, shutil, glob
from glinker.rag.ingestion import build_index
import glinker.config as cfg

WORKING_INDEX = cfg.RAG_INDEX_DIR
DATASET_MOUNT = '/kaggle/input/datasets/hassanraheem/uresense-rag-index'

if os.path.exists(f"{WORKING_INDEX}/index.faiss"):
    print("Index already in working dir — skipping.")

else:
    # Search for index.faiss at any depth inside the dataset mount
    matches = glob.glob(f"{DATASET_MOUNT}/**/index.faiss", recursive=True)
    if not matches:
        matches = glob.glob(f"{DATASET_MOUNT}/index.faiss")

    if matches:
        source_dir = os.path.dirname(matches[0])
        print(f"Pre-built dataset found at: {source_dir} — copying to working dir ...")
        os.makedirs(WORKING_INDEX, exist_ok=True)
        for fname in ("index.faiss", "chunks.json"):
            shutil.copy(f"{source_dir}/{fname}", f"{WORKING_INDEX}/{fname}")
        print("Copied. Cell 6 will load it.")

    else:
        print(f"index.faiss not found under {DATASET_MOUNT}")
        print(f"Contents: {os.listdir(DATASET_MOUNT) if os.path.exists(DATASET_MOUNT) else 'dataset not mounted'}")
        print("Building from scratch (20-40 min) ...")
        build_index(textbooks_limit=5000, guidelines_limit=2000)
        print("Index built and saved to", WORKING_INDEX)


In [ ]:
# ── Cell 6: Load RAG index into memory ───────────────────────────────────────
from glinker.rag.retrieval import load_index
load_index()


In [ ]:
# ── Cell 7: Load disease ranking datasets (optional) ─────────────────────────
# Requires the 9 Kaggle symptom-disease datasets attached via Add Data.
# The pipeline degrades gracefully (no ranking) if none are attached.
from glinker.disease.ranker import load_datasets
load_datasets()


In [ ]:
# ── Cell 8: Start Flask on port 5001 (GPU 0) ─────────────────────────────────
#
# Whisper + GLiNER were loaded onto GPU 0 in Cell 4.
# Flask is started here WITHOUT a ngrok tunnel — the single shared ngrok tunnel
# is created in Cell 9 on the reverse proxy (port 5003) which routes to Flask.
#
import threading
from api.app import app as flask_app

_flask_thread = threading.Thread(
    target=lambda: flask_app.run(port=5001, debug=False, use_reloader=False),
    daemon=True,
)
_flask_thread.start()
print("Flask started on port 5001 (no tunnel yet — Cell 9 handles ngrok)")

In [ ]:
# ── Cell 9: MedGemma (GPU 1) + reverse proxy + single ngrok tunnel ───────────
#
# Proxy on port 5003 routes internally:
#   /analyze/*    → MedGemma FastAPI on 5002
#   everything else → Flask on 5001
#
# ngrok is explicitly disconnected before reconnecting so the static domain
# always points to the proxy (5003), not a stale tunnel from a previous run.
#
import socket, threading, uvicorn, httpx
from fastapi import FastAPI, Request
from fastapi.responses import Response
from pyngrok import ngrok
from api.medgemma_app import app as medgemma_app, _load_model

# ── 1. Start MedGemma on port 5002 (skip if already running) ─────────────────
def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) == 0

if _port_in_use(5002):
    print("[MedGemma] Already running on port 5002 — skipping restart")
else:
    threading.Thread(
        target=lambda: uvicorn.run(medgemma_app, host="0.0.0.0", port=5002, log_level="warning"),
        daemon=True,
    ).start()
    print("[MedGemma] FastAPI started on port 5002")

# ── 2. Pre-warm model (VRAM logs printed inside _load_model) ──────────────────
_load_model()

# ── 3. Reverse proxy on port 5003 ────────────────────────────────────────────
_proxy = FastAPI()

@_proxy.api_route("/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "PATCH"])
async def _route(path: str, request: Request):
    body    = await request.body()
    headers = {k: v for k, v in request.headers.items() if k.lower() != "host"}
    port    = 5002 if path.startswith("analyze/") else 5001
    async with httpx.AsyncClient(timeout=300.0) as client:
        resp = await client.request(
            request.method, f"http://localhost:{port}/{path}",
            content=body, headers=headers,
        )
    return Response(content=resp.content, status_code=resp.status_code,
                    headers=dict(resp.headers))

if _port_in_use(5003):
    print("[Proxy] Already running on port 5003 — skipping restart")
else:
    threading.Thread(
        target=lambda: uvicorn.run(_proxy, host="0.0.0.0", port=5003, log_level="warning"),
        daemon=True,
    ).start()
    print("[Proxy] Started on port 5003")

import time; time.sleep(2)  # let servers bind before ngrok connects

# ── 4. Disconnect any existing tunnels, then connect to proxy ─────────────────
for t in ngrok.get_tunnels():
    print(f"[ngrok] Disconnecting old tunnel: {t.public_url} → {t.config['addr']}")
    ngrok.disconnect(t.public_url)

PUBLIC_URL = ngrok.connect(5003).public_url
print(f"[ngrok] Connected → {PUBLIC_URL} → proxy:5003")

print("\n" + "=" * 64)
print(f"  AI_SERVICE_URL       = {PUBLIC_URL}")
print(f"  MEDGEMMA_SERVICE_URL = {PUBLIC_URL}")
print("=" * 64)
print("\nBoth point to the same URL — proxy routes internally.")
print("Paste both into Backend/.env and restart Express.")